In [12]:
import numpy as np
from typing import List, Dict, Tuple, Optional
import pandas as pd
import talib

def purged_split(labels:pd.DataFrame, max_timeframe_lookback_window:int, test_date: str, embargo_hour:int = 4):
    labels = labels.set_index(pd.to_datetime(labels['t0']))

    limit_date = pd.to_datetime(test_date).tz_localize("Asia/Taipei")  - pd.Timedelta(hours=max_timeframe_lookback_window*4+embargo_hour)

    labels_train_val = labels[labels.index < limit_date]
    labels_test = labels[labels.index >= limit_date]

    val_start_date = limit_date - pd.Timedelta(days = 60) 
    val_limit_date = val_start_date - pd.Timedelta(hours=max_timeframe_lookback_window*4+embargo_hour)
    labels_train = labels_train_val[labels_train_val.index < val_limit_date]
    labels_val = labels_train_val[labels_train_val.index >= val_start_date]
    print(f"Train: {labels_train.shape}, Val: {labels_val.shape}, Test: {labels_test.shape}")
    assert labels_train.index.max() < labels_val.index.min()-pd.Timedelta(hours=max_timeframe_lookback_window*4+embargo_hour), "Training and validation sets overlap!"
    assert labels_val.index.max() < labels_test.index.min()-pd.Timedelta(hours=max_timeframe_lookback_window*4+embargo_hour), "Validation and test sets overlap!"
    return labels_train, labels_val, labels_test


def _range_ewm_vol(df: pd.DataFrame, halflife: int = 20) -> pd.Series:
    """
    OHL-range 的指數加權移動平均（EWM）波動率。
    Formula: range = (high - low) / open

    Args:
        - df:pd.DataFrame, 必須包含 'open','high','low' 欄位
        - halflife:int, 指數加權平滑的半衰期

    Returns: 
        - 相對波動率:pd.Series, 經過指數加權平滑處理，波動率序列，與 df 同索引，向前移動一格
    """
    if not all(col in df.columns for col in ['open', 'high', 'low']):
        raise ValueError("DataFrame must contain 'open', 'high', and 'low' columns.")
    rng = (df['high'] - df['low']) / df['open'].replace(0.0, np.nan)
    return rng.ewm(halflife=halflife, adjust=False).mean().shift(1)


def compute_features(data:pd.DataFrame) -> pd.DataFrame:
    """
    計算技術指標特徵
    Args:
        data: DataFrame with columns ['open', 'high', 'low', 'close', 'volume']
    Returns:
        data: DataFrame with additional feature columns
    """
    if not all(col in data.columns for col in ['open', 'high', 'low', 'close', 'volume']):
        raise ValueError("DataFrame must contain 'open', 'high', 'low', 'close', and 'volume' columns.")
    returns = data['close'].pct_change().shift(-1)

    #Add Features
    data[f"obv"] = talib.OBV(data['close'], data['volume'])
    data[f"volatility"] = _range_ewm_vol(data, halflife=20)
    
    data[f"momentum"] = talib.ATR(data['high'], data['low'], data['close'], timeperiod=14)

    for n in [5,10,20]:
        data[f"ema_{n}"] = talib.EMA(data['close'], timeperiod=n)
        data[f"rsi_{n}"] = talib.RSI(data['close'], timeperiod=n)
        data[f"atr_{n}"] = talib.ATR(data['high'], data['low'], data['close'], timeperiod=n)
        data[f"cci_{n}"] = talib.CCI(data['high'], data['low'], data['close'], timeperiod=n)
        data[f"mfi_{n}"] = talib.MFI(data['high'], data['low'], data['close'], data['volume'], timeperiod=n)
        data[f"adx_{n}"] = talib.ADX(data['high'], data['low'], data['close'], timeperiod=n)
        data[f"willr_{n}"] = talib.WILLR(data['high'], data['low'], data['close'], timeperiod=n)
    data.fillna(0, inplace=True)
    return data
def data_pipeline(label:pd.DataFrame,df_1h:pd.DataFrame, df_4h:pd.DataFrame) -> Tuple[Dict[str, np.ndarray], np.ndarray]:
    """
    數據處理管線: 由於label 是事件，需要從事件時間點往前取K線數據
    1. 讀取1小時和4小時K線數據
    2. 計算技術指標特徵
    3. 根據label的timestamp往前取K線數據
    4. 返回多時間框架的OHLCV數據字典
    Args:
        label: DataFrame with columns ['timestamp', 'open', 'high', 'low', 'close', 'volume']
    Returns:
        data_dict: {'1h': np.ndarray, '4h': np.ndarray}, 每個key對應的值是形狀為 (num_samples, num_timesteps, num_features) 的numpy陣列
        labels: np.ndarray, 形狀為 (num_samples,) 的標籤陣列
    """
    # try:
    #     df_1h = pd.read_csv('../data/1h_klines.csv',index_col=['datetime'],parse_dates=['datetime'])
    #     df_4h = pd.read_csv('../data/4h_klines.csv',index_col=['datetime'],parse_dates=['datetime'])

    # except FileNotFoundError:
    #     raise FileNotFoundError("Kline data files not found. Please ensure 'data/1h_klines.csv' and 'data/4h_klines.csv' exist.")
    
    # try:
    #     df_1h = compute_features(df_1h)
    #     df_4h = compute_features(df_4h)
    # except Exception as e:
    #     raise ValueError(f"Error computing features: {e}")
    
    data_dict = {}
    for _,row in label.iterrows():
        timestamp = pd.to_datetime(row['t0'])
        # 取1小時K線
        start_time_1h = timestamp - pd.Timedelta(hours=100)  # 假設取100個1小時K線
        mask_1h = (df_1h.index > start_time_1h) & (df_1h.index < timestamp)
        data_1h = df_1h.loc[mask_1h]

        print(f"Timestamp: {timestamp}, 1h data shape: {data_1h.shape}")
        if len(data_1h) < 100:
            # padding
            padding = pd.DataFrame(np.zeros((100 - len(data_1h), data_1h.shape[1])), columns=data_1h.columns)
            data_1h = pd.concat([padding, data_1h], ignore_index=True)
            print(f"Warning: Not enough 1h data before {timestamp}. Expected at least 100, got {len(data_1h)}")
        
        # 取4小時K線
        start_time_4h = timestamp - pd.Timedelta(hours=25*4)  # 假設取25個4小時K線
        mask_4h = (df_4h.index > start_time_4h) & (df_4h.index < timestamp)
        data_4h = df_4h.loc[mask_4h]
        print(f"Timestamp: {timestamp}, 4h data shape: {data_4h.shape}")
        if len(data_4h) < 25:
            # padding
            padding = pd.DataFrame(np.zeros((25 - len(data_4h), data_4h.shape[1])), columns=data_4h.columns)
            data_4h = pd.concat([padding, data_4h], ignore_index=True)
            print(f"Warning: Not enough 4h data before {timestamp}. Expected at least 25, got {len(data_4h)}")
        
        if '1h' not in data_dict:
            data_dict['1h'] = []
        if '4h' not in data_dict:
            data_dict['4h'] = []
        
        data_dict['1h'].append(data_1h.values)
        data_dict['4h'].append(data_4h.values)
    # 將list轉為np.ndarray
    data_dict['1h'] = np.array(data_dict['1h'])
    data_dict['4h'] = np.array(data_dict['4h'])
    return data_dict, label['label'].values

In [14]:
label = pd.read_csv('../data/label.csv')
label_train, label_val, label_test = purged_split(label, max_timeframe_lookback_window=25, test_date='2025-04-30 23:00:00')
print(label_train.head(3))
print(label_val.head(3))
print(label_test.head(3))
try:
    df_1h = pd.read_csv('../data/1h_klines.csv',index_col=['datetime'],parse_dates=['datetime'])
    df_4h = pd.read_csv('../data/4h_klines.csv',index_col=['datetime'],parse_dates=['datetime'])

except FileNotFoundError:
    raise FileNotFoundError("Kline data files not found. Please ensure 'data/1h_klines.csv' and 'data/4h_klines.csv' exist.")

try:
    df_1h = compute_features(df_1h)
    df_4h = compute_features(df_4h)
except Exception as e:
    raise ValueError(f"Error computing features: {e}")

Train: (746, 8), Val: (53, 8), Test: (116, 8)
                                                  t0  side  entry_price  \
t0                                                                        
2023-01-02 12:00:00+08:00  2023-01-02 12:00:00+08:00     1      16654.7   
2023-01-03 08:00:00+08:00  2023-01-03 08:00:00+08:00    -1      16665.9   
2023-01-05 22:00:00+08:00  2023-01-05 22:00:00+08:00    -1      16771.0   

                                                  t1  label            pt  \
t0                                                                          
2023-01-02 12:00:00+08:00  2023-01-02 15:00:00+08:00    1.0  16726.511949   
2023-01-03 08:00:00+08:00  2023-01-03 14:00:00+08:00    0.0  16602.720439   
2023-01-05 22:00:00+08:00  2023-01-05 23:00:00+08:00    0.0  16726.199498   

                                     sl       vol  
t0                                                 
2023-01-02 12:00:00+08:00  16601.043439  0.002151  
2023-01-03 08:00:00+08:00  16750.513

In [31]:
data, label_data = data_pipeline(label, df_1h, df_4h)
print(f"label : {len(label_data)}, type: {type(label_data)}\n{label_data}")

Timestamp: 2023-01-02 12:00:00+08:00, 1h data shape: (36, 30)
Timestamp: 2023-01-02 12:00:00+08:00, 4h data shape: (9, 30)
Timestamp: 2023-01-03 08:00:00+08:00, 1h data shape: (56, 30)
Timestamp: 2023-01-03 08:00:00+08:00, 4h data shape: (14, 30)
Timestamp: 2023-01-05 22:00:00+08:00, 1h data shape: (99, 30)
Timestamp: 2023-01-05 22:00:00+08:00, 4h data shape: (25, 30)
Timestamp: 2023-01-06 00:00:00+08:00, 1h data shape: (99, 30)
Timestamp: 2023-01-06 00:00:00+08:00, 4h data shape: (24, 30)
Timestamp: 2023-01-09 00:00:00+08:00, 1h data shape: (99, 30)
Timestamp: 2023-01-09 00:00:00+08:00, 4h data shape: (24, 30)
Timestamp: 2023-01-09 01:00:00+08:00, 1h data shape: (99, 30)
Timestamp: 2023-01-09 01:00:00+08:00, 4h data shape: (25, 30)
Timestamp: 2023-01-10 17:00:00+08:00, 1h data shape: (99, 30)
Timestamp: 2023-01-10 17:00:00+08:00, 4h data shape: (25, 30)
Timestamp: 2023-01-17 02:00:00+08:00, 1h data shape: (99, 30)
Timestamp: 2023-01-17 02:00:00+08:00, 4h data shape: (25, 30)
Timestamp

In [28]:
print(f"1h data shape: {data['1h'].shape}, 4h data shape: {data['4h'].shape}")
print(data['1h'][0])

1h data shape: (919, 100, 30), 4h data shape: (919, 25, 30)
[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [ 1.67262120e+12  1.65780000e+04  1.65793000e+04 ...  5.90216721e+01
   0.00000000e+00 -4.76299694e+01]
 [ 1.67262480e+12  1.65565000e+04  1.65871000e+04 ...  5.92542697e+01
   0.00000000e+00 -2.93577982e+01]
 [ 1.67262840e+12  1.65804000e+04  1.67007000e+04 ...  7.07195417e+01
   0.00000000e+00 -2.26091221e+01]]
